# Part 1: Foundation Model Exploration

This notebook compares a zero-training Hugging Face sentiment model with the HW2 proactive customer satisfaction model.

Important context from the assignment: the HW2 model predicts satisfaction before the review is written using order and delivery features, so it is proactive. The foundation model reads the review text after the customer writes it, so it is reactive.

In [ ]:
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from common import (
    FEATURES, TARGET, build_order_level_dataset, find_data_path, load_raw_data,
)

pd.set_option('display.max_columns', 100)
DATA_PATH = find_data_path()
print(f'Detected dataset: {DATA_PATH.name}')

## Load Olist Data and Check Review Text Availability

The assignment requires `review_comment_message`. The cleaned HW2/HW3 modeling dataset in this folder contains `review_score_mean` and review dates, but it does not contain the text field. The code below supports either case: if a review-text file is supplied through `REVIEW_TEXT_PATH`, it merges that file by `order_id`; otherwise it reports the limitation clearly instead of inventing review text.

In [ ]:
raw = load_raw_data(DATA_PATH)
print(raw.shape)
print([c for c in raw.columns if 'review' in c.lower() or 'comment' in c.lower() or 'message' in c.lower()])

review_text_path = os.getenv('REVIEW_TEXT_PATH')
if 'review_comment_message' not in raw.columns and review_text_path:
    review_text = pd.read_csv(review_text_path)
    keep_cols = [c for c in ['order_id', 'review_score', 'review_comment_message'] if c in review_text.columns]
    raw = raw.merge(review_text[keep_cols], on='order_id', how='left')

text_col = 'review_comment_message' if 'review_comment_message' in raw.columns else None
if text_col is None:
    print('No review_comment_message column is available in the detected cleaned modeling dataset.')
else:
    print(raw[text_col].notna().sum(), 'records have non-null review text')

## Build the Same HW2 Order-Level Features

This reuses the HW2 aggregation, target, feature engineering, and serialized Random Forest pipeline.

In [ ]:
agg = build_order_level_dataset(raw)
print(agg.shape)
print('Positive rate:', agg[TARGET].mean())
agg[FEATURES + [TARGET, 'review_score_mean']].head()

## Foundation Model Inference on 500 Non-Empty Reviews

Model required by the assignment: `nlptown/bert-base-multilingual-uncased-sentiment`. It predicts 1-5 star labels from multilingual text. Predicted 4-5 stars are mapped to positive; 1-3 stars are mapped to negative.

In [ ]:
if text_col is None:
    foundation_sample = pd.DataFrame()
    print('SKIPPED: The local cleaned dataset has no review text. Supply REVIEW_TEXT_PATH with order_id and review_comment_message to run this cell.')
else:
    from transformers import pipeline

    text_rows = raw[raw[text_col].fillna('').astype(str).str.strip().ne('')].copy()
    text_rows = text_rows.drop_duplicates('order_id')
    foundation_sample = text_rows.sample(n=min(500, len(text_rows)), random_state=42).copy()

    sentiment = pipeline(
        'sentiment-analysis',
        model='nlptown/bert-base-multilingual-uncased-sentiment',
    )

    outputs = sentiment(foundation_sample[text_col].astype(str).tolist(), truncation=True)
    foundation_sample['foundation_label'] = [o['label'] for o in outputs]
    foundation_sample['foundation_confidence'] = [o['score'] for o in outputs]
    foundation_sample['foundation_stars'] = foundation_sample['foundation_label'].str.extract(r'(\d+)').astype(int)
    foundation_sample['foundation_pred'] = (foundation_sample['foundation_stars'] >= 4).astype(int)
    foundation_sample['actual'] = (foundation_sample['review_score_mean'] >= 4).astype(int)

    display(foundation_sample[['order_id', text_col, 'review_score_mean', 'actual', 'foundation_label', 'foundation_confidence', 'foundation_pred']].head())

## Class Distribution

In [ ]:
if foundation_sample.empty:
    print('Class distribution cannot be calculated without non-empty review text records.')
else:
    class_distribution = pd.DataFrame({
        'actual_count': foundation_sample['actual'].value_counts().sort_index(),
        'foundation_pred_count': foundation_sample['foundation_pred'].value_counts().sort_index(),
    }).rename(index={0: 'negative', 1: 'positive'})
    display(class_distribution)

## HW2 Model Predictions on the Same 500 Records

The HW2 model is evaluated on the exact same sampled orders. This is still not an identical business use case: the HW2 model is proactive and uses order features available before a review is written, while the foundation model is reactive and uses the final review text.

In [ ]:
model_path = Path('model/model.pkl')
if not model_path.exists():
    print('model/model.pkl not found. Run: python train_and_serialize.py')
elif foundation_sample.empty:
    print('HW2 comparison on the same 500 records requires the review-text sample above.')
else:
    hw2_model = joblib.load(model_path)
    sample_orders = foundation_sample[['order_id', 'actual']].merge(agg, on='order_id', how='left')
    sample_orders['hw2_proba'] = hw2_model.predict_proba(sample_orders[FEATURES])[:, 1]
    sample_orders['hw2_pred'] = (sample_orders['hw2_proba'] >= 0.5).astype(int)
    display(sample_orders[['order_id', 'actual', 'hw2_pred', 'hw2_proba']].head())

## Required Comparison Table

In [ ]:
def binary_metrics(y_true, y_pred):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1 Score': f1_score(y_true, y_pred, zero_division=0),
    }

if foundation_sample.empty or not Path('model/model.pkl').exists():
    comparison = pd.DataFrame({
        'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
        'Foundation Model (review text)': ['Requires review_comment_message'] * 4,
        'Your HW2 Model (order features)': ['Requires same 500 review-text records'] * 4,
    })
else:
    fm = binary_metrics(foundation_sample['actual'], foundation_sample['foundation_pred'])
    hw2 = binary_metrics(sample_orders['actual'], sample_orders['hw2_pred'])
    comparison = pd.DataFrame({
        'Metric': list(fm.keys()),
        'Foundation Model (review text)': list(fm.values()),
        'Your HW2 Model (order features)': [hw2[k] for k in fm.keys()],
    })

display(comparison)

## Reflection

The HW2 model and the foundation model solve related but different operational problems. The HW2 Random Forest is a proactive early-warning model: it uses order, payment, product, and delivery features to estimate whether a customer is likely to leave a positive review before the review text exists. That makes it useful for intervention, routing, service recovery, and monitoring fulfillment quality. The foundation model is reactive: it reads the written review text after the customer has already expressed an opinion, so it is better suited for labeling, triage, summarization, and post-hoc voice-of-customer analysis.

If review text is available, the foundation model may perform strongly because its input directly contains customer sentiment. Its advantage is speed to deployment: it requires no Olist-specific training data and can handle Portuguese text out of the box. The tradeoff is computational cost, dependency on a large transformer download, and weaker control over domain-specific errors. The custom HW2 model requires feature engineering and training, but it is cheaper to serve, aligned to the exact business definition, and available before the review arrives.

In production, I would use both. The HW2 model would flag risky orders proactively, while the foundation model would classify review text after delivery to enrich monitoring labels, audit false positives and false negatives, and create new features for future retraining.